# SHAPE Agent 16 — Student-centered Handheld Architecture for Programming Education

**Tagline:** Learning through interaction. Improving through feedback.

**Course:** ITAI2376, Houston Community College — **Group:** ITAI2376_14562
**Proposal deadline:** November 13, 2026 · **Final deadline:** December 7, 2026

**Team:** Michael Rivas (Integration and Project Coordination) · Cherluk (Learning Design and Instructional Experience) · Olu (Agent Behavior and Interaction Engineering) · Taki (Evaluation, Testing, and Learning Analytics)
Full names for Cherluk, Olu and Taki are editable fields in the Team Workspace tab — they are **not** invented here.

**What this notebook is:** a complete, working SHAPE Agent 16 application in Google Colab. Run the cells top to bottom. The last cell launches the app. You do **not** need to paste an API key into this notebook — see the Architecture section below.

**How to use it (beginner path):**
1. Run the setup cell (installs). Wait for it to finish.
2. Run the core cells. They define the lessons, tools, agents and policy.
3. Run the interface cell. It builds the app but does not launch it yet.
4. Run the last cell. The app opens below it.
5. In the app: Learning Lab → pick a lesson → submit code → check Agent Workspace → check Learner Progress and Adaptive Learning.


In [ ]:
# Setup cell — install dependencies if missing, check Python (quiet pip).
# Run this first in a FRESH Colab session. It is safe to re-run.
import importlib.util, subprocess, sys
print("Python:", sys.version)
for _pkg in ("gradio", "pandas", "matplotlib"):
    if importlib.util.find_spec(_pkg) is None:
        print(f"Installing {_pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg])
    else:
        print(f"{_pkg} already installed")
import gradio, pandas, matplotlib
print("OK: gradio", gradio.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)


## Architecture — how SHAPE Agent 16 thinks

One LLM serves four agent roles (they are prompts + tool routing, **not** four separate LLM services):

- **Instructor Agent** — explains concepts in small steps, with examples.
- **Debugger Agent** — explains *why* an error happened, gives graduated hints.
- **Evaluator Agent** — checks work against the expected output, pass/fail.
- **Learning Strategist Agent** — picks the next step from the learner's recent results.

A central **orchestrator** decides which role and which tool should act.

### The 8-step reasoning cycle (every learner message goes through this)
1. **Receive and validate** learner input.
2. **Retrieve** relevant learner and lesson context.
3. **Plan** the next instructional action.
4. **Select** an agent role and a tool.
5. **Execute** the action.
6. **Evaluate** the result.
7. **Update** learner memory and the feedback policy.
8. **Explain** the outcome to the learner.

You can watch all 8 steps in the **Agent Workspace** tab — that transparency is a course requirement, not a decoration.

### LLM vs local fallback — read this before you demo
- With an API key, SHAPE calls OpenAI (`gpt-4o-mini`) through a plain `urllib` request. No extra SDK is needed.
- **How to give it a key in Colab (never paste it in this notebook):** Colab left panel → Secrets (key icon) → add a secret named `OPENAI_API_KEY` → enable "Notebook access". Or set the environment variable `OPENAI_API_KEY` before launching.
- **Without a key**, SHAPE uses local template replies and the app says so in plain text: *"Local fallback — no API key found."* The fallback is **never** presented as an LLM.
- Features that need a key: LLM-generated explanations and hints. Everything else — lessons, code checking, progress, policy, reports — works without a key.


## Core 1 — Data store, lessons and the HCC resource library
This cell creates `shape.db` (SQLite, one database name used everywhere), the 6 Python-fundamentals lessons, the 5-question pre/post assessment, and the 12 HCC resources with their exact URLs. The HCC pages may need an HCC login; SHAPE does **not** claim to have accessed them.


In [ ]:
"""SHAPE Agent 16 — data store, lessons and HCC resource index.

SQLite database: shape.db (one name, used consistently everywhere).
Tables: learners, attempts, lessons_done, contributions, policy_state.
"""
import json
import sqlite3
from datetime import date

DB_PATH = "shape.db"

# ---------------------------------------------------------------------------
# HCC resource library — exact URLs/titles from the project brief.
# These pages may require an authenticated HCC account. SHAPE does NOT claim
# to have accessed or extracted content from any restricted page.
# ---------------------------------------------------------------------------
RESOURCES = [
    {"title": "DL Intro", "topic": "Deep Learning Introduction",
     "url": "https://eagleonline.hccs.edu/courses/340456/modules/items/24356286", "notes": "", "source": ""},
    {"title": "DL Toolkit", "topic": "Deep Learning Toolkit",
     "url": "https://eagleonline.hccs.edu/courses/340456/modules/items/24356292", "notes": "", "source": ""},
    {"title": "NN Basics", "topic": "Neural Network Basics",
     "url": "https://eagleonline.hccs.edu/courses/340456/modules/items/24356301", "notes": "", "source": ""},
    {"title": "CNN", "topic": "Convolutional Neural Networks",
     "url": "https://eagleonline.hccs.edu/courses/340456/modules/items/25011377", "notes": "", "source": ""},
    {"title": "Sequence Modeling and NLP", "topic": "Sequence Modeling and Natural Language Processing",
     "url": "https://eagleonline.hccs.edu/courses/340456/modules/items/24356319", "notes": "", "source": ""},
    {"title": "Applications of Transformers and Attention Mechanics", "topic": "Transformers and Attention",
     "url": "https://eagleonline.hccs.edu/courses/340456/modules/items/24356329", "notes": "", "source": ""},
    {"title": "Autoencoders and Applied Variational (VAEs)", "topic": "Autoencoders and VAEs",
     "url": "https://eagleonline.hccs.edu/courses/340456/modules/items/25301433", "notes": "", "source": ""},
    {"title": "Generative Adversarial Networks GANs", "topic": "GANs",
     "url": "https://eagleonline.hccs.edu/courses/340456/modules/items/24356349", "notes": "", "source": ""},
    {"title": "Gen AI with Diffusion Models", "topic": "Diffusion Models",
     "url": "https://eagleonline.hccs.edu/courses/340456/modules/items/24356356", "notes": "", "source": ""},
    {"title": "Reinforcement Learning in AI Agents", "topic": "Reinforcement Learning",
     "url": "https://eagleonline.hccs.edu/courses/340456/modules/items/24356369", "notes": "", "source": ""},
    {"title": "Advanced Agents and Reasoning Models", "topic": "Advanced Agents",
     "url": "https://eagleonline.hccs.edu/courses/340456/modules/items/24356381", "notes": "", "source": ""},
    {"title": "Agent Planning and Orchestration", "topic": "Agent Planning",
     "url": "https://eagleonline.hccs.edu/courses/340456/modules/items/24356400", "notes": "", "source": ""},
]

# Learning pathway (resources -> Agent 16 progression).
PATHWAY = [
    ("1. Deep learning foundations", ["DL Intro", "DL Toolkit"]),
    ("2. Neural networks and CNNs", ["NN Basics", "CNN"]),
    ("3. Sequence modeling and NLP", ["Sequence Modeling and NLP"]),
    ("4. Transformers and attention", ["Applications of Transformers and Attention Mechanics"]),
    ("5. Autoencoders and generative models", ["Autoencoders and Applied Variational (VAEs)", "Generative Adversarial Networks GANs"]),
    ("6. Diffusion models", ["Gen AI with Diffusion Models"]),
    ("7. Reinforcement learning", ["Reinforcement Learning in AI Agents"]),
    ("8. Advanced agents, reasoning, planning, and orchestration", ["Advanced Agents and Reasoning Models", "Agent Planning and Orchestration"]),
]

# ---------------------------------------------------------------------------
# Lessons — Python fundamentals first, before advanced AI-agent concepts.
# ---------------------------------------------------------------------------
LESSONS = [
    {"id": "L1", "title": "Print and Variables",
     "concept": "print() shows text or a value on the screen. A variable is a named box that stores a value, e.g. name = 'Ada'. Use print(name) to see what is inside.",
     "example": "name = 'Ada'\nprint('Hello,', name)",
     "example_output": "Hello, Ada",
     "exercise": "Write code that prints exactly this line:\nHello, SHAPE",
     "expected_output": "Hello, SHAPE",
     "hints": ["Use print() with the text in quotes.",
               "Try: print('Hello, SHAPE') — the comma and spelling must match exactly.",
               "The whole answer is one line: print(\"Hello, SHAPE\")"],
     "concept_check_q": "What does print('Hi') do?",
     "concept_check_a": "shows Hi on the screen"},
    {"id": "L2", "title": "Input and Types",
     "concept": "Every value has a type: int (whole number), float (decimal), str (text), bool (True/False). type() tells you the type. input() always returns a str, so convert it with int() or float() before doing maths.",
     "example": "print(type(42))\nprint(type(3.5))\nprint(type('hi'))",
     "example_output": "<class 'int'>\n<class 'float'>\n<class 'str'>",
     "exercise": "Write code that prints the type of 42.\nExpected output is exactly:\n<class 'int'>",
     "expected_output": "<class 'int'>",
     "hints": ["Use the type() function, then print the result.",
               "Try: print(type(42))",
               "The whole answer is one line: print(type(42))"],
     "concept_check_q": "What type does input() return, even if the user types a number?",
     "concept_check_a": "str"},
    {"id": "L3", "title": "Conditionals",
     "concept": "if / elif / else lets code choose. The condition is True or False. Indentation (4 spaces) shows which lines belong to each branch.",
     "example": "score = 85\nif score >= 80:\n    print('pass')\nelse:\n    print('retry')",
     "example_output": "pass",
     "exercise": "Set x = 7. If x is greater than 5, print 'big', otherwise print 'small'.\nExpected output: big",
     "expected_output": "big",
     "hints": ["Start with x = 7, then an if statement: if x > 5:",
               "Indent the print('big') line by 4 spaces under the if, and add else: with print('small').",
               "x = 7\nif x > 5:\n    print('big')\nelse:\n    print('small')"],
     "concept_check_q": "What does an if statement need after its condition?",
     "concept_check_a": "a colon and an indented block"},
    {"id": "L4", "title": "Loops",
     "concept": "A for loop repeats code for each item. range(3) gives 0, 1, 2. A while loop repeats while a condition is True — make sure it can become False, or it never stops.",
     "example": "for i in range(3):\n    print(i)",
     "example_output": "0\n1\n2",
     "exercise": "Write a loop that prints the numbers 0, 1, 2, each on its own line.",
     "expected_output": "0\n1\n2",
     "hints": ["range(3) gives 0, 1, 2.",
               "Try: for i in range(3): then print(i) on the next, indented line.",
               "for i in range(3):\n    print(i)"],
     "concept_check_q": "What values does range(3) produce?",
     "concept_check_a": "0, 1, 2"},
    {"id": "L5", "title": "Functions",
     "concept": "A function is a named, reusable block: def name(inputs): ... return value. Calling it runs the block. return sends a value back; print only shows it.",
     "example": "def double(n):\n    return n * 2\n\nprint(double(4))",
     "example_output": "8",
     "exercise": "Define a function double(n) that returns n * 2, then print double(4).\nExpected output: 8",
     "expected_output": "8",
     "hints": ["Start with def double(n): and return n * 2 on the indented line.",
               "After the function, add print(double(4)).",
               "def double(n):\n    return n * 2\n\nprint(double(4))"],
     "concept_check_q": "What is the difference between return and print in a function?",
     "concept_check_a": "return sends a value back to the caller; print only shows it"},
    {"id": "L6", "title": "Lists",
     "concept": "A list holds several values in order: nums = [1, 2, 3]. len() counts them, nums[0] is the first item, and sum() adds numbers. Lists can be changed after they are made.",
     "example": "nums = [1, 2, 3]\nprint(len(nums))\nprint(sum(nums))",
     "example_output": "3\n6",
     "exercise": "Make a list nums = [1, 2, 3], then print the sum of its items.\nExpected output: 6",
     "expected_output": "6",
     "hints": ["First make the list: nums = [1, 2, 3]",
               "Then print the sum: print(sum(nums))",
               "nums = [1, 2, 3]\nprint(sum(nums))"],
     "concept_check_q": "What is the first item of [10, 20, 30], and how do you get it?",
     "concept_check_a": "10, with [0]"},
]

# 5-question pre/post assessment (same questions both times, so change is measurable).
ASSESSMENT = [
    {"q": "What does print('Hi') output?", "options": ["Hi", "print", "'Hi'", "Nothing"], "answer": "Hi"},
    {"q": "What is the type of 42 in Python?", "options": ["str", "float", "int", "bool"], "answer": "int"},
    {"q": "x = 7. What does this print? if x > 5: print('big') else: print('small')", "options": ["small", "big", "7", "Error"], "answer": "big"},
    {"q": "How many times does for i in range(3): print(i) print?", "options": ["2", "3", "4", "0"], "answer": "3"},
    {"q": "def double(n): return n * 2 — what is double(4)?", "options": ["4", "6", "8", "n * 2"], "answer": "8"},
]

TEAM = [
    {"first": "Michael Rivas", "full_name": "Michael Rivas", "role": "Integration and Project Coordination"},
    {"first": "Cherluk", "full_name": "", "role": "Learning Design and Instructional Experience"},
    {"first": "Olu", "full_name": "", "role": "Agent Behavior and Interaction Engineering"},
    {"first": "Taki", "full_name": "", "role": "Evaluation, Testing, and Learning Analytics"},
]
MEMBER_NAMES = [m["first"] for m in TEAM]

CONTRIBUTION_FIELDS = ["member", "full_name", "date", "workstream", "category",
                       "task", "contribution", "evidence", "evidence_link",
                       "decisions", "challenges", "outcome", "reflection"]
CATEGORIES = ["Design", "Coding", "Testing", "Reflection"]


def get_lesson(lesson_id):
    for lesson in LESSONS:
        if lesson["id"] == lesson_id:
            return lesson
    return LESSONS[0]


def get_resource(title):
    for r in RESOURCES:
        if r["title"] == title:
            return r
    return None


def init_db(db_path=DB_PATH):
    """Create shape.db and all tables if they do not exist. Safe to re-run."""
    con = sqlite3.connect(db_path)
    cur = con.cursor()
    cur.execute("""CREATE TABLE IF NOT EXISTS learners
        (learner_id TEXT PRIMARY KEY, created TEXT, last_lesson TEXT)""")
    cur.execute("""CREATE TABLE IF NOT EXISTS attempts
        (id INTEGER PRIMARY KEY AUTOINCREMENT, learner_id TEXT, lesson_id TEXT,
         code TEXT, passed INTEGER, hints_used INTEGER, reward REAL,
         strategy TEXT, created TEXT)""")
    cur.execute("""CREATE TABLE IF NOT EXISTS lessons_done
        (learner_id TEXT, lesson_id TEXT, completed TEXT,
         PRIMARY KEY (learner_id, lesson_id))""")
    cur.execute("""CREATE TABLE IF NOT EXISTS contributions
        (id INTEGER PRIMARY KEY AUTOINCREMENT, member TEXT, full_name TEXT,
         date TEXT, workstream TEXT, category TEXT, task TEXT, contribution TEXT,
         evidence TEXT, evidence_link TEXT, decisions TEXT, challenges TEXT,
         outcome TEXT, reflection TEXT)""")
    cur.execute("""CREATE TABLE IF NOT EXISTS policy_state
        (strategy TEXT PRIMARY KEY, q REAL, updated TEXT)""")
    con.commit()
    con.close()
    return db_path


def record_attempt(db_path, learner_id, lesson_id, code, passed, hints_used, reward, strategy):
    con = sqlite3.connect(db_path)
    cur = con.cursor()
    cur.execute("INSERT OR IGNORE INTO learners VALUES (?,?,?)",
                (learner_id, date.today().isoformat(), lesson_id))
    cur.execute("""INSERT INTO attempts
        (learner_id, lesson_id, code, passed, hints_used, reward, strategy, created)
        VALUES (?,?,?,?,?,?,?,?)""",
                (learner_id, lesson_id, code, 1 if passed else 0, hints_used,
                 reward, strategy, date.today().isoformat()))
    if passed:
        cur.execute("INSERT OR IGNORE INTO lessons_done VALUES (?,?,?)",
                    (learner_id, lesson_id, date.today().isoformat()))
    con.commit()
    con.close()


def get_attempts(db_path, learner_id=None):
    con = sqlite3.connect(db_path)
    con.row_factory = sqlite3.Row
    cur = con.cursor()
    if learner_id:
        cur.execute("SELECT * FROM attempts WHERE learner_id=? ORDER BY id", (learner_id,))
    else:
        cur.execute("SELECT * FROM attempts ORDER BY id")
    rows = [dict(r) for r in cur.fetchall()]
    con.close()
    return rows


def progress_summary(db_path, learner_id):
    attempts = get_attempts(db_path, learner_id)
    if not attempts:
        return {"attempts": 0, "correct": 0, "accuracy": 0.0, "hints": 0,
                "lessons_done": 0, "total_reward": 0.0}
    correct = sum(a["passed"] for a in attempts)
    return {"attempts": len(attempts), "correct": correct,
            "accuracy": round(100.0 * correct / len(attempts), 1),
            "hints": sum(a["hints_used"] for a in attempts),
            "lessons_done": len({a["lesson_id"] for a in attempts if a["passed"]}),
            "total_reward": round(sum(a["reward"] for a in attempts), 1)}


def save_contribution(db_path, record):
    con = sqlite3.connect(db_path)
    cur = con.cursor()
    cols = [c for c in CONTRIBUTION_FIELDS if c in record]
    vals = [record.get(c, "") for c in cols]
    cur.execute(f"INSERT INTO contributions ({','.join(cols)}) VALUES ({','.join('?' * len(cols))})", vals)
    con.commit()
    row_id = cur.lastrowid
    con.close()
    return row_id


def get_contributions(db_path, member=None):
    con = sqlite3.connect(db_path)
    con.row_factory = sqlite3.Row
    cur = con.cursor()
    if member:
        cur.execute("SELECT * FROM contributions WHERE member=? ORDER BY id", (member,))
    else:
        cur.execute("SELECT * FROM contributions ORDER BY id")
    rows = [dict(r) for r in cur.fetchall()]
    con.close()
    return rows


def save_policy(db_path, q_table):
    con = sqlite3.connect(db_path)
    cur = con.cursor()
    for strategy, q in q_table.items():
        cur.execute("INSERT OR REPLACE INTO policy_state VALUES (?,?,?)",
                    (strategy, float(q), date.today().isoformat()))
    con.commit()
    con.close()


def load_policy(db_path, defaults):
    con = sqlite3.connect(db_path)
    cur = con.cursor()
    cur.execute("SELECT strategy, q FROM policy_state")
    loaded = dict(cur.fetchall())
    con.close()
    out = dict(defaults)
    out.update({k: float(v) for k, v in loaded.items() if k in out})
    return out


def to_json(obj):
    return json.dumps(obj, indent=2)


## Core 2 — Adaptive teaching policy (reinforcement-learning concept)
A small Q-value / contextual-bandit policy over 5 teaching strategies. Reward: correct with no hints +10, correct with 1 hint +7, correct with multiple hints +4, failed attempt −3, concept-check failed −1. Update: `Q[s] += 0.2 × (reward − Q[s])`, epsilon-greedy 0.1. A reward increase is **not** proof of learning — learning is measured separately by the pre/post assessment.


In [ ]:
"""SHAPE Agent 16 — adaptive teaching policy (contextual bandit / Q-value).

Five teaching strategies. The policy learns from learner feedback, but a
reward increase is NOT proof of learning — learning is evaluated
independently with the pre/post assessment (see Evaluation Plan).
"""
import random

STRATEGIES = ["Worked example", "Concept explanation", "Small hint",
              "Simpler practice", "Challenge exercise"]
ALPHA = 0.2          # learning rate: Q[s] += ALPHA * (reward - Q[s])
EPSILON = 0.1        # epsilon-greedy exploration


def new_q_table():
    return {s: 0.0 for s in STRATEGIES}


def reward_for(passed, hints_used, concept_check_failed=False):
    """Documented reward function (from the project brief)."""
    if concept_check_failed:
        return -1.0
    if not passed:
        return -3.0
    if hints_used == 0:
        return 10.0
    if hints_used == 1:
        return 7.0
    return 4.0


def update_q(q_table, strategy, reward, alpha=ALPHA):
    """Q[s] += alpha * (reward - Q[s]). Returns (new_q, old_q)."""
    old = float(q_table.get(strategy, 0.0))
    new = old + alpha * (reward - old)
    q_table[strategy] = round(new, 3)
    return q_table[strategy], old


def select_strategy(q_table, rng=None, epsilon=EPSILON):
    """Epsilon-greedy: mostly the best-known strategy, sometimes explore.
    Ties are broken by the fixed STRATEGIES order, so results are stable."""
    rng = rng or random
    if rng.random() < epsilon:
        choice = rng.choice(STRATEGIES)
        return choice, f"Exploration (epsilon={epsilon}): tried '{choice}' to learn how it works for this learner."
    best = max(STRATEGIES, key=lambda s: q_table.get(s, 0.0))
    return best, (f"Exploitation: '{best}' has the highest Q-value "
                  f"({q_table.get(best, 0.0):.2f}) from past rewards.")


def strategy_for_lesson(lesson_id, q_table, rng=None):
    """Pick a strategy, with a sensible starting point per lesson."""
    choice, reason = select_strategy(q_table, rng=rng)
    return choice, reason


def q_table_rows(q_table):
    return [[s, round(float(q_table.get(s, 0.0)), 3)] for s in STRATEGIES]


## Core 3 — Tools (at least two, with validation, errors and fallbacks)
**Tool 1 — `run_code_tool`:** runs learner code in a restricted namespace (AST check, only `math`/`random` imports, no `open`/`eval`/`exec`/`os`/`sys`, no file access) and compares stdout to the expected output.
**Plain-language limit:** this is a *classroom exercise runner*, **not** a hardened security sandbox. Do not use it to run arbitrary untrusted public code, and never describe it as secure.
**Tool 2 — `retrieve_lesson_tool`:** searches SHAPE's own curated lesson + HCC resource index by keyword. No web access is used. The orchestrator logs every tool call (name, input, output) in the Agent Workspace tab.


In [ ]:
"""SHAPE Agent 16 — tools.

Tool 1: run_code_tool — controlled Python exercise runner.
Tool 2: retrieve_lesson_tool — lesson / HCC resource retrieval.

LIMITATION, stated plainly: the AST check + restricted namespace below is
a CLASSROOM exercise runner, not a hardened security sandbox. It blocks
the obvious escapes (imports other than math/random, open/eval/exec,
os/sys/subprocess, file access, dunder access), but it must NOT be used
to run arbitrary untrusted public code. Never describe it as secure.
"""
import ast
import io
import sys
from contextlib import redirect_stdout

# In-notebook: LESSONS, RESOURCES and get_lesson are defined in the store cell above.

ALLOWED_IMPORTS = {"math", "random"}
BANNED_NAMES = {"open", "eval", "exec", "compile", "__import__", "input",
                "os", "sys", "subprocess", "socket", "pathlib", "globals",
                "locals", "vars", "getattr", "setattr", "delattr", "breakpoint"}
ALLOWED_BUILTINS = {"print": print, "len": len, "range": range, "str": str,
                    "int": int, "float": float, "bool": bool, "list": list,
                    "dict": dict, "set": set, "tuple": tuple, "sum": sum,
                    "min": min, "max": max, "abs": abs, "round": round,
                    "enumerate": enumerate, "zip": zip, "sorted": sorted,
                    "reversed": reversed, "type": type,
                    # __import__ is needed so `import math` / `import random`
                    # (the only imports the AST check allows) can run at all.
                    # Learner code may NOT call __import__ directly: the AST
                    # check bans the name '__import__', and only math/random
                    # pass the import check, so this cannot be used to reach
                    # os/sys/subprocess.
                    "__import__": __import__}


def _check_code(code):
    """Return an error string, or None if the code passes the checks."""
    if not isinstance(code, str) or not code.strip():
        return "No code submitted. Write some Python code first."
    if len(code) > 5000:
        return "Code is too long (over 5000 characters)."
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return f"SyntaxError: {e.msg} (line {e.lineno})"
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                if alias.name.split(".")[0] not in ALLOWED_IMPORTS:
                    return f"Import not allowed: '{alias.name}'. Only math and random are allowed."
        if isinstance(node, ast.ImportFrom):
            if (node.module or "").split(".")[0] not in ALLOWED_IMPORTS:
                return f"Import not allowed: '{node.module}'. Only math and random are allowed."
        if isinstance(node, ast.Name) and node.id in BANNED_NAMES:
            return f"Not allowed in exercises: '{node.id}'."
        if isinstance(node, ast.Attribute) and node.attr.startswith("__"):
            return "Dunder (double-underscore) access is not allowed."
        if isinstance(node, (ast.Global, ast.Nonlocal)):
            return "global / nonlocal are not allowed in exercises."
    return None


def run_code_tool(code, expected_output=None):
    """Run learner code in a restricted namespace and compare stdout.

    Returns dict: passed, stdout, expected, error, hint.
    The runner is a classroom aid, NOT a hardened sandbox.
    """
    error = _check_code(code)
    if error:
        return {"passed": False, "stdout": "", "expected": expected_output or "",
                "error": error, "hint": "Fix the problem above, then submit again."}
    buf = io.StringIO()
    safe_globals = {"__builtins__": dict(ALLOWED_BUILTINS)}
    try:
        with redirect_stdout(buf):
            exec(compile(code, "<exercise>", "exec"), safe_globals)
    except Exception as e:  # noqa: BLE001 — learner errors are the point
        return {"passed": False, "stdout": buf.getvalue(),
                "expected": expected_output or "",
                "error": f"{type(e).__name__}: {e}",
                "hint": "Read the error message: it usually names the line and the problem."}
    stdout = buf.getvalue()
    if expected_output is None:
        return {"passed": True, "stdout": stdout, "expected": "",
                "error": "", "hint": "It ran without an error."}
    passed = stdout.strip() == str(expected_output).strip()
    hint = ""
    if not passed:
        hint = (f"Your output was {stdout.strip()!r}, expected {str(expected_output).strip()!r}. "
                "Check spelling, spacing and line breaks — they must match exactly.")
    return {"passed": passed, "stdout": stdout, "expected": str(expected_output),
            "error": "", "hint": hint}


def retrieve_lesson_tool(query):
    """Search the curated lesson + HCC resource index by keyword.

    Returns dict: found, lesson (or None), resources (list), message.
    No web access is used; this searches only the project's own index.
    """
    if not isinstance(query, str) or not query.strip():
        return {"found": False, "lesson": None, "resources": [],
                "message": "Give me a topic to look up, e.g. 'loops' or 'functions'."}
    q = query.strip().lower()
    lesson = None
    for item in LESSONS:
        hay = (item["id"] + " " + item["title"] + " " + item["concept"]).lower()
        if q in hay or any(word in hay for word in q.split() if len(word) > 3):
            lesson = {"id": item["id"], "title": item["title"],
                      "concept": item["concept"], "example": item["example"]}
            break
    resources = [r for r in RESOURCES if q in (r["title"] + " " + r["topic"]).lower()
                 or any(word in (r["title"] + " " + r["topic"]).lower()
                        for word in q.split() if len(word) > 3)]
    if lesson is None and not resources:
        return {"found": False, "lesson": None, "resources": [],
                "message": f"No lesson or HCC resource matched '{query}'. Try: print, variables, loops, functions, lists, CNN, transformers, reinforcement learning."}
    return {"found": True, "lesson": lesson, "resources": resources,
            "message": "Found in the SHAPE lesson / HCC resource index (HCC pages may need an HCC login)."}


## Core 4 — LLM client, agent roles, orchestrator, reports and hardware mock
`orchestrate()` is the 8-step cycle in code. `generate_report_draft()` builds an individual report **only** from that member's recorded contributions — it flags missing evidence and unanswered reflections and never invents work. `handle_device_message()` validates the handheld-device JSON contract `{"device_id","event","lesson_id","action","timestamp"}`.


In [ ]:
"""SHAPE Agent 16 — LLM client, four agent roles, orchestrator, reports, hardware mock.

One LLM, four roles. Each role is a prompt + a tool-routing decision, not
a separate LLM service. If no API key is available, every role falls back
to local template logic and the UI says so — the fallback is NEVER
presented as an LLM.
"""
import json
import os
import urllib.error
import urllib.request
from datetime import datetime

# In-notebook: reward_for, select_strategy, update_q are defined in the policy cell above.
# In-notebook: CONTRIBUTION_FIELDS, LESSONS, MEMBER_NAMES, get_lesson are defined above.
# In-notebook: retrieve_lesson_tool and run_code_tool are defined in the tools cell above.

LLM_MODEL = "gpt-4o-mini"
LLM_TIMEOUT = 20


def get_api_key():
    """API key from Colab secrets first, then the environment. Never from
    notebook source code, and never printed or logged."""
    try:
        from google.colab import userdata  # type: ignore
        key = userdata.get("OPENAI_API_KEY")
        if key:
            return key, "Colab secret OPENAI_API_KEY"
    except Exception:
        pass
    key = os.environ.get("OPENAI_API_KEY", "")
    if key:
        return key, "environment variable OPENAI_API_KEY"
    return "", ""


def llm_mode():
    key, source = get_api_key()
    if key:
        return f"LLM active (OpenAI {LLM_MODEL}, key from {source})", True
    return "Local fallback — no API key found. Responses are local templates, NOT an LLM.", False


def call_llm(system_prompt, user_prompt, api_key=None):
    """Plain urllib call to OpenAI chat completions. Any failure -> None
    (caller must use the local fallback). Returns the reply text or None."""
    key = api_key or get_api_key()[0]
    if not key:
        return None
    payload = json.dumps({
        "model": LLM_MODEL,
        "messages": [{"role": "system", "content": system_prompt},
                     {"role": "user", "content": user_prompt}],
        "temperature": 0.3, "max_tokens": 400,
    }).encode("utf-8")
    req = urllib.request.Request(
        "https://api.openai.com/v1/chat/completions", data=payload,
        headers={"Authorization": f"Bearer {key}", "Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=LLM_TIMEOUT) as resp:
            data = json.loads(resp.read().decode("utf-8"))
        return data["choices"][0]["message"]["content"].strip()
    except Exception:
        return None


# --- Agent roles (prompts; one LLM serves all four) -------------------------
ROLE_SYSTEM = {
    "Instructor": "You are SHAPE's Instructor Agent. Explain simply, in short steps, with one small example. Never do the exercise for the learner.",
    "Debugger": "You are SHAPE's Debugger Agent. Explain why an error happened in plain language, then give one small hint at a time. Never just give the corrected code.",
    "Evaluator": "You are SHAPE's Evaluator Agent. Check work against the expected output, state clearly pass or fail, and say exactly what differed.",
    "Learning Strategist": "You are SHAPE's Learning Strategist Agent. Recommend the next step (review, practice, or challenge) and give one reason based on the learner's recent results.",
}

FALLBACK = {
    "Instructor": "Here is the idea in small steps: {detail} Try the worked example first, then the exercise.",
    "Debugger": "The error was: {detail} Read the line it names, check spelling and indentation, and try again — one fix at a time.",
    "Evaluator": "{detail}",
    "Learning Strategist": "Next step: {detail} This is based on your recent attempts, not a guess.",
}


def agent_reply(role, detail, context=""):
    """Ask the LLM for this role; on any failure use the local template.
    Returns (text, mode) where mode is 'LLM' or 'Local fallback'."""
    mode_text, has_key = llm_mode()
    if has_key:
        text = call_llm(ROLE_SYSTEM[role], f"Context: {context}\nDetail: {detail}")
        if text:
            return text, "LLM"
    return FALLBACK[role].format(detail=detail), "Local fallback"


def route_intent(text):
    """Orchestrator routing by intent (keyword rules, documented)."""
    t = (text or "").lower()
    if any(w in t for w in ("error", "debug", "bug", "traceback", "wrong", "fail")):
        return "Debugger"
    if any(w in t for w in ("check", "evaluate", "grade", "correct?", "did i", "is this right")):
        return "Evaluator"
    if any(w in t for w in ("next", "plan", "strategy", "what should i", "review", "challenge")):
        return "Learning Strategist"
    return "Instructor"


def orchestrate(learner_id, message, lesson_id="L1", code=None, hints_used=0,
                q_table=None, rng=None):
    """The 8-step reasoning cycle, in code:
    1 receive/validate, 2 retrieve context, 3 plan, 4 select agent+tool,
    5 execute, 6 evaluate, 7 update memory/policy, 8 explain.
    Returns a dict the UI displays in the Agent Workspace tab."""
    log = []
    # 1. Receive and validate.
    if not isinstance(message, str) or not message.strip():
        return {"agent": "Instructor", "decision": "Input was empty.",
                "tool_calls": [], "response": "Please type a question, or submit some code.",
                "mode": "Local fallback", "log": ["1. Input empty — asked learner to try again."]}
    log.append("1. Received and validated learner input.")
    # 2. Retrieve context.
    lesson = get_lesson(lesson_id)
    log.append(f"2. Retrieved context: lesson {lesson['id']} ({lesson['title']}), learner {learner_id}.")
    # 3. Plan.
    role = route_intent(message)
    log.append(f"3. Planned: route to {role} Agent (intent from learner's words).")
    # 4. Select agent + tool.
    tool_calls = []
    tool_result = None
    if code is not None:
        tool_calls.append({"tool": "run_code_tool", "input": code[:80]})
        tool_result = run_code_tool(code, lesson["expected_output"])
        log.append(f"4. Selected tool run_code_tool for submitted code (passed={tool_result['passed']}).")
    elif any(w in message.lower() for w in ("lesson", "learn", "what is", "resource", "topic")):
        tool_calls.append({"tool": "retrieve_lesson_tool", "input": message})
        tool_result = retrieve_lesson_tool(message)
        log.append(f"4. Selected tool retrieve_lesson_tool for a topic question (found={tool_result['found']}).")
    else:
        log.append("4. No tool needed for this message.")
    # 5. Execute + 6. Evaluate.
    if tool_result is not None and code is not None:
        detail = (f"Your code {'passed' if tool_result['passed'] else 'did not pass'}. "
                  f"Output: {tool_result['stdout'].strip()!r}. "
                  f"Expected: {tool_result['expected']!r}. {tool_result.get('hint', '')}")
        if tool_result.get("error"):
            role = "Debugger"
            detail = tool_result["error"] + " " + tool_result.get("hint", "")
    elif tool_result is not None:
        detail = tool_result["message"]
        if tool_result.get("lesson"):
            detail += f" Lesson: {tool_result['lesson']['title']} — {tool_result['lesson']['concept']}"
    else:
        detail = lesson["concept"]
    log.append("5. Executed the action.")
    log.append("6. Evaluated the result.")
    # 7. Update memory/policy (only when a code result exists and a policy was given).
    policy_note = "No policy update for this message."
    if q_table is not None and code is not None and tool_result is not None:
        reward = reward_for(tool_result["passed"], hints_used)
        strategy, reason = select_strategy(q_table, rng=rng)
        before = q_table.get(strategy, 0.0)
        update_q(q_table, strategy, reward)
        policy_note = (f"Reward {reward:+.0f} for this attempt. Strategy '{strategy}' "
                       f"Q {before:.2f} -> {q_table[strategy]:.2f}. {reason}")
        log.append(f"7. Updated policy: {policy_note}")
    else:
        log.append("7. No policy update (no code result in this message).")
    # 8. Explain.
    response, mode = agent_reply(role, detail, context=f"Lesson {lesson['id']}: {lesson['title']}")
    log.append(f"8. Explained the outcome to the learner (via {role} Agent, {mode}).")
    return {"agent": role, "decision": f"Routed to {role} Agent. {policy_note}",
            "tool_calls": tool_calls, "response": response,
            "mode": mode, "log": log, "tool_result": tool_result}


# --- Individual report drafts (from actual records only) --------------------
def generate_report_draft(member, contributions):
    """Draft a member's individual report from THEIR contribution records.
    Never fabricates. Flags missing evidence and unanswered reflections."""
    mine = [c for c in contributions if c.get("member") == member]
    lines = [f"# SHAPE Agent 16 — Individual Report Draft: {member}",
             "",
             "DRAFT — generated from contribution records in shape.db. "
             "The member must review and correct this draft before submission. "
             "Nothing in this draft was invented.",
             "",
             f"Contributions recorded: {len(mine)}",
             ""]
    if not mine:
        lines += ["## Missing evidence",
                  "",
                  f"No contribution records were found for {member}. "
                  "Add records in the Team Workspace tab before this report can be completed.",
                  ""]
        return "\n".join(lines)
    by_cat = {}
    for c in mine:
        by_cat.setdefault(c.get("category", "Uncategorised"), []).append(c)
    lines.append("## Contributions by category")
    lines.append("")
    for cat in ["Design", "Coding", "Testing", "Reflection"]:
        items = by_cat.get(cat, [])
        lines.append(f"- {cat}: {len(items)} record(s)")
    lines.append("")
    for c in mine:
        lines.append(f"### {c.get('date', '')} — {c.get('task', '')} ({c.get('category', '')})")
        lines.append("")
        lines.append(f"- Workstream: {c.get('workstream', '')}")
        lines.append(f"- Contribution: {c.get('contribution', '')}")
        lines.append(f"- Evidence: {c.get('evidence', '')} {c.get('evidence_link', '')}")
        lines.append(f"- Technical decisions: {c.get('decisions', '')}")
        lines.append(f"- Challenges: {c.get('challenges', '')}")
        lines.append(f"- Outcome: {c.get('outcome', '')}")
        lines.append(f"- Reflection: {c.get('reflection', '')}")
        lines.append("")
    missing_ev = [c for c in mine if not (c.get("evidence") or "").strip()]
    missing_refl = [c for c in mine if not (c.get("reflection") or "").strip()]
    lines.append("## Missing evidence / unanswered sections")
    lines.append("")
    lines.append(f"- Records with no evidence description: {len(missing_ev)}")
    lines.append(f"- Records with no reflection: {len(missing_refl)}")
    if missing_ev:
        lines.append("- Add an evidence description / link for: " +
                     "; ".join(c.get("task", "?") for c in missing_ev))
    if missing_refl:
        lines.append("- Add a reflection for: " +
                     "; ".join(c.get("task", "?") for c in missing_refl))
    lines.append("")
    lines.append("## Shared team work vs individual work")
    lines.append("")
    lines.append("This draft lists only this member's recorded contributions. "
                 "Shared accomplishments (the Colab app, the lesson set, the "
                 "evaluation plan) belong to the whole team and are described "
                 "in the team report, not claimed here as individual work.")
    return "\n".join(lines)


def team_evidence_summary(contributions):
    lines = ["# SHAPE Agent 16 — Team Evidence Summary", "",
             f"Total contribution records: {len(contributions)}", ""]
    for member in MEMBER_NAMES:
        mine = [c for c in contributions if c.get("member") == member]
        cats = {}
        for c in mine:
            cats[c.get("category", "?")] = cats.get(c.get("category", "?"), 0) + 1
        lines.append(f"- {member}: {len(mine)} record(s) — " +
                     (", ".join(f"{k}: {v}" for k, v in sorted(cats.items())) or "no records"))
    lines.append("")
    lines.append("Missing evidence overall: " +
                 str(sum(1 for c in contributions if not (c.get("evidence") or "").strip())))
    return "\n".join(lines)


# --- Hardware mock ------------------------------------------------------------
def handle_device_message(msg):
    """Validate a handheld-device JSON message. Contract:
    {"device_id","event","lesson_id","action","timestamp"}.
    The host runs the LLM; devices only send small events. No hardware needed."""
    required = ["device_id", "event", "lesson_id", "action", "timestamp"]
    if not isinstance(msg, dict):
        return {"ok": False, "error": "Message must be a JSON object."}
    missing = [k for k in required if k not in msg or msg[k] in ("", None)]
    if missing:
        return {"ok": False, "error": f"Missing fields: {', '.join(missing)}."}
    return {"ok": True, "ack": f"Received '{msg['event']}' from {msg['device_id']} "
                               f"for lesson {msg['lesson_id']}: action={msg['action']}. "
                               "Host will respond with the next prompt (no LLM runs on the device).",
            "received_at": datetime.now().isoformat(timespec="seconds")}


## Safety boundaries
- **Input validation:** empty code, over-long code, syntax errors and banned imports are rejected with a plain-language message, not a crash.
- **Code runner:** restricted namespace only (see Core 3). No file, network or operating-system access from learner code.
- **No personal data:** learner IDs are anonymised codes like `learner-01`. No real names are required anywhere.
- **Fallbacks:** LLM failure → local fallback (labelled). Tool failure → an error message + a hint, not a silent pass.
- **Transparency:** the Agent Workspace shows the agent, the decision, the tools used and whether the reply was LLM or fallback.


## Interface — the Gradio app (Gradio 6)
This cell **builds** the app (it does not launch it). Tabs: **A** Learning Lab · **B** Agent Workspace · **C** Learner Progress · **D** Adaptive Learning · **E** Team Workspace · **F** Export Center. Every button has a working callback. The launch cell uses `theme=gr.themes.Soft()` (Gradio 6 puts theme on launch(), not on Blocks()) — no deprecated theme/CSS constructor arguments.


In [ ]:
"""SHAPE Agent 16 — Gradio interface (Gradio 6 compatible).

Uses gr.Blocks() with no constructor theme/CSS args (Gradio 6 moved
theme to launch(): the notebook's last cell calls
demo.launch(share=False, theme=gr.themes.Soft())).
All buttons have working callbacks. The UI is built by build_demo();
the notebook's last cell launches it.
"""
import csv
import io
import json
import os
import random

import gradio as gr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

# In-notebook: agent functions are defined in the agents cell above.
# In-notebook: policy functions are defined in the policy cell above.
# In-notebook: store data and functions are defined in the store cell above.
# In-notebook: tools are defined in the tools cell above.

DB = "shape.db"


def _lesson_choices():
    return [(f"{l['id']} — {l['title']}", l["id"]) for l in LESSONS]


def _progress_figure(learner_id):
    attempts = get_attempts(DB, learner_id)
    fig, ax = plt.subplots(figsize=(6, 3))
    if not attempts:
        ax.text(0.5, 0.5, "No attempts yet — submit code in the Learning Lab.",
                ha="center", va="center")
        ax.set_title("Learner progress")
    else:
        xs = list(range(1, len(attempts) + 1))
        ax.plot(xs, [a["passed"] for a in attempts], marker="o", label="Correct (1=yes, 0=no)")
        ax.plot(xs, [a["hints_used"] for a in attempts], marker="s", label="Hints used")
        ax.set_xlabel("Attempt number")
        ax.set_ylabel("Count")
        ax.set_title(f"Progress for {learner_id} ({len(attempts)} attempts)")
        ax.set_ylim(-0.2, max(2, max(a["hints_used"] for a in attempts) + 1))
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    fig.tight_layout()
    return fig


def _synthetic_figure():
    """SYNTHETIC DEMO ONLY — never real student results."""
    fig, ax = plt.subplots(figsize=(6, 3))
    groups = ["Fixed instruction", "Adaptive Agent 16"]
    pre = [52, 53]
    post = [61, 74]
    x = range(len(groups))
    ax.bar([i - 0.2 for i in x], pre, 0.4, label="Pre-test (synthetic)")
    ax.bar([i + 0.2 for i in x], post, 0.4, label="Post-test (synthetic)")
    ax.set_xticks(list(x)); ax.set_xticklabels(groups)
    ax.set_ylabel("Mean score (%)")
    ax.set_title("SYNTHETIC DEMO ONLY — fixed vs adaptive (made-up data)")
    ax.set_ylim(0, 100)
    ax.legend(fontsize=8)
    fig.tight_layout()
    return fig


def _attempts_df(learner_id):
    rows = get_attempts(DB, learner_id)
    if not rows:
        return pd.DataFrame(columns=["lesson", "passed", "hints", "reward", "strategy"])
    return pd.DataFrame([{"lesson": r["lesson_id"], "passed": r["passed"],
                          "hints": r["hints_used"], "reward": r["reward"],
                          "strategy": r["strategy"]} for r in rows])


def _contrib_df():
    rows = get_contributions(DB)
    if not rows:
        return pd.DataFrame(columns=CONTRIBUTION_FIELDS)
    return pd.DataFrame([{k: r.get(k, "") for k in CONTRIBUTION_FIELDS} for r in rows])


def build_demo():
    init_db(DB)
    q_table = load_policy(DB, new_q_table())
    policy_before = dict(q_table)
    mode_text, _ = llm_mode()

    with gr.Blocks(title="SHAPE Agent 16") as demo:
        gr.Markdown("# SHAPE Agent 16 — Learning through interaction. Improving through feedback.")
        gr.Markdown(f"**Course:** ITAI2376, Houston Community College, Group ITAI2376_14562 · "
                    f"**Final deadline:** Dec 7, 2026 · **Proposal:** Nov 13, 2026\n\n"
                    f"**Mode:** {mode_text}  \n"
                    "If no API key is set, the app uses local fallback templates and says so — "
                    "the fallback is never presented as an LLM.")
        hint_state = gr.State(0)
        pre_state = gr.State(None)

        with gr.Tabs():
            # ---- A. Learning Lab ------------------------------------------------
            with gr.Tab("A. Learning Lab"):
                learner_id = gr.Textbox(label="Learner ID (anonymised — use a code like learner-01, no real names)",
                                        value="learner-01")
                lesson_dd = gr.Dropdown(choices=_lesson_choices(), value="L1", label="Lesson")
                concept_box = gr.Textbox(label="Concept", lines=4)
                example_box = gr.Code(label="Worked example", language="python", lines=5)
                exercise_box = gr.Textbox(label="Exercise", lines=3)
                code_box = gr.Code(label="Your code", language="python", lines=6)
                with gr.Row():
                    submit_btn = gr.Button("Submit code", variant="primary")
                    hint_btn = gr.Button("Get a hint")
                feedback_box = gr.Textbox(label="Feedback", lines=5)
                hint_box = gr.Textbox(label="Hint", lines=3)
                check_q = gr.Textbox(label="Concept check", lines=2)
                check_a = gr.Textbox(label="Your concept-check answer")
                with gr.Row():
                    check_btn = gr.Button("Check concept answer")
                check_out = gr.Textbox(label="Concept-check result", lines=2)
                gr.Markdown("### Pre / post assessment (same 5 questions — change is measurable)")
                assess_inputs = []
                for i, item in enumerate(ASSESSMENT):
                    assess_inputs.append(gr.Radio(choices=item["options"], label=f"Q{i+1}. {item['q']}"))
                with gr.Row():
                    pre_btn = gr.Button("Score pre-test")
                    post_btn = gr.Button("Score post-test")
                assess_out = gr.Textbox(label="Assessment result", lines=3)

                def on_lesson(lesson_id):
                    lesson = get_lesson(lesson_id)
                    return (lesson["concept"], lesson["example"], lesson["exercise"],
                            lesson["concept_check_q"], "", "", 0)

                lesson_dd.change(on_lesson, inputs=[lesson_dd],
                                 outputs=[concept_box, example_box, exercise_box, check_q,
                                          feedback_box, hint_box, hint_state])

                def on_submit(learner, lesson_id, code, hints):
                    lesson = get_lesson(lesson_id)
                    result = orchestrate(learner, "check my code", lesson_id=lesson_id,
                                         code=code, hints_used=int(hints or 0), q_table=q_table)
                    reward = reward_for(result["tool_result"]["passed"], int(hints or 0)) if result.get("tool_result") else 0.0
                    record_attempt(DB, learner, lesson_id, code,
                                   result["tool_result"]["passed"] if result.get("tool_result") else False,
                                   int(hints or 0), reward, result["agent"])
                    save_policy(DB, q_table)
                    passed = result["tool_result"]["passed"] if result.get("tool_result") else False
                    text = (f"{'Correct!' if passed else 'Not yet correct.'} "
                            f"Output: {result['tool_result']['stdout'].strip()!r} "
                            f"Expected: {result['tool_result']['expected']!r} "
                            f"Reward: {reward:+.0f}. {result['response']}")
                    return text, 0

                submit_btn.click(on_submit, inputs=[learner_id, lesson_dd, code_box, hint_state],
                                 outputs=[feedback_box, hint_state])

                def on_hint(lesson_id, hints):
                    lesson = get_lesson(lesson_id)
                    n = int(hints or 0)
                    if n >= len(lesson["hints"]):
                        return "No more hints for this exercise — try the worked example again.", n
                    return f"Hint {n+1} of {len(lesson['hints'])}: {lesson['hints'][n]}", n + 1

                hint_btn.click(on_hint, inputs=[lesson_dd, hint_state],
                               outputs=[hint_box, hint_state])

                def on_check(lesson_id, answer):
                    lesson = get_lesson(lesson_id)
                    ok = lesson["concept_check_a"].lower() in (answer or "").lower() or (answer or "").lower() in lesson["concept_check_a"].lower()
                    if ok:
                        return "Concept check: correct."
                    return f"Concept check: not quite. Expected idea: {lesson['concept_check_a']}. Reward -1 if this were scored."

                check_btn.click(on_check, inputs=[lesson_dd, check_a], outputs=[check_out])

                def on_pre(*answers):
                    score = sum(1 for a, item in zip(answers, ASSESSMENT) if a == item["answer"])
                    return (f"Pre-test: {score}/5 correct. Keep this score — "
                            "take the post-test after the lessons."), score

                def on_post(*args):
                    *answers, pre = args
                    score = sum(1 for a, item in zip(answers, ASSESSMENT) if a == item["answer"])
                    if pre is None:
                        return (f"Post-test: {score}/5 correct. No pre-test score was recorded "
                                "in this session, so improvement cannot be measured yet."), None
                    change = score - int(pre)
                    return (f"Post-test: {score}/5 correct. Pre-test was {pre}/5. "
                            f"Change: {change:+d} question(s). This measures test change, "
                            "not proof of learning on its own."), pre

                pre_btn.click(on_pre, inputs=assess_inputs, outputs=[assess_out, pre_state])
                post_btn.click(on_post, inputs=assess_inputs + [pre_state],
                               outputs=[assess_out, pre_state])

            # ---- B. Agent Workspace ----------------------------------------------
            with gr.Tab("B. Agent Workspace"):
                gr.Markdown("See which agent is responding, what tool it used, and whether the reply came from the LLM or the local fallback.")
                ws_learner = gr.Textbox(label="Learner ID", value="learner-01")
                ws_lesson = gr.Dropdown(choices=_lesson_choices(), value="L1", label="Lesson context")
                ws_msg = gr.Textbox(label="Ask SHAPE anything (try: 'explain loops', 'debug my code', 'what should I do next?')", lines=3)
                ws_code = gr.Code(label="Optional code to check (leave blank for a question)", language="python", lines=4)
                ask_btn = gr.Button("Ask the orchestrator", variant="primary")
                ws_agent = gr.Textbox(label="Agent role responding")
                ws_decision = gr.Textbox(label="Instructional decision", lines=2)
                ws_mode = gr.Textbox(label="LLM or local fallback?")
                ws_tools = gr.Textbox(label="Tool activity", lines=3)
                ws_log = gr.Textbox(label="Reasoning-cycle log (8 steps)", lines=8)
                ws_resp = gr.Textbox(label="Response", lines=5)

                def on_ask(learner, lesson_id, message, code):
                    code_arg = code if code and code.strip() else None
                    out = orchestrate(learner, message, lesson_id=lesson_id, code=code_arg, q_table=q_table)
                    tools = "\n".join(f"{t['tool']}: {t['input']}" for t in out["tool_calls"]) or "No tool used."
                    return (out["agent"], out["decision"], out["mode"], tools,
                            "\n".join(out["log"]), out["response"])

                ask_btn.click(on_ask, inputs=[ws_learner, ws_lesson, ws_msg, ws_code],
                              outputs=[ws_agent, ws_decision, ws_mode, ws_tools, ws_log, ws_resp])

            # ---- C. Learner Progress ----------------------------------------------
            with gr.Tab("C. Learner Progress"):
                prog_learner = gr.Textbox(label="Learner ID", value="learner-01")
                refresh_btn = gr.Button("Refresh progress")
                prog_summary = gr.Markdown()
                prog_df = gr.Dataframe(label="Attempts", interactive=False)
                prog_plot = gr.Plot(label="Progress chart")

                def on_refresh(learner):
                    s = progress_summary(DB, learner)
                    md = (f"**Attempts:** {s['attempts']} · **Correct:** {s['correct']} · "
                          f"**Accuracy:** {s['accuracy']}% · **Hints used:** {s['hints']} · "
                          f"**Lessons completed:** {s['lessons_done']} / {len(LESSONS)} · "
                          f"**Total reward:** {s['total_reward']}")
                    return md, _attempts_df(learner), _progress_figure(learner)

                refresh_btn.click(on_refresh, inputs=[prog_learner],
                                  outputs=[prog_summary, prog_df, prog_plot])

            # ---- D. Adaptive Learning ---------------------------------------------
            with gr.Tab("D. Adaptive Learning"):
                gr.Markdown("A reward increase is **not** proof of learning. Learning is measured "
                            "independently with the pre/post assessment in the Learning Lab.")
                adapt_refresh = gr.Button("Refresh policy")
                adapt_table = gr.Dataframe(label="Q-table: strategy vs Q-value (before -> now)",
                                           headers=["Strategy", "Q before", "Q now"], interactive=False)
                adapt_strategy = gr.Textbox(label="Strategy the policy would select next, and why", lines=3)
                adapt_plot = gr.Plot(label="Synthetic evaluation (demo data only)")

                def on_adapt():
                    rows = [[s, round(float(policy_before.get(s, 0.0)), 3),
                             round(float(q_table.get(s, 0.0)), 3)] for s in STRATEGIES]
                    choice, reason = select_strategy(q_table, rng=random.Random(0))
                    return (pd.DataFrame(rows, columns=["Strategy", "Q before", "Q now"]),
                            f"Selected: {choice}. {reason}", _synthetic_figure())

                adapt_refresh.click(on_adapt, inputs=[], outputs=[adapt_table, adapt_strategy, adapt_plot])

            # ---- E. Team Workspace --------------------------------------------------
            with gr.Tab("E. Team Workspace"):
                gr.Markdown("Every member records Design, Coding, Testing and Reflection work here. "
                            "Full names for Cherluk, Olu and Taki are editable fields — they are not invented here.")
                tw_member = gr.Dropdown(choices=MEMBER_NAMES, label="Member (first name as listed)")
                tw_full = gr.Textbox(label="Full name (editable — leave blank if not yet confirmed)")
                tw_date = gr.Textbox(label="Date (YYYY-MM-DD)", value="2026-11-06")
                tw_stream = gr.Textbox(label="Workstream", value="SHAPE Agent 16")
                tw_cat = gr.Dropdown(choices=CATEGORIES, label="Category")
                tw_task = gr.Textbox(label="Task completed")
                tw_contrib = gr.Textbox(label="Specific contribution", lines=3)
                tw_evidence = gr.Textbox(label="Evidence description", lines=2)
                tw_link = gr.Textbox(label="Evidence link or filename")
                tw_decisions = gr.Textbox(label="Technical decisions", lines=2)
                tw_challenges = gr.Textbox(label="Challenges encountered", lines=2)
                tw_outcome = gr.Textbox(label="Outcome", lines=2)
                tw_reflection = gr.Textbox(label="Individual reflection", lines=3)
                with gr.Row():
                    tw_save = gr.Button("Save contribution", variant="primary")
                    tw_refresh = gr.Button("Refresh list")
                tw_status = gr.Textbox(label="Status", lines=2)
                tw_df = gr.Dataframe(label="Contribution log", interactive=False)
                tw_file = gr.File(label="Import contributions CSV (columns: member, full_name, date, workstream, category, task, contribution, evidence, evidence_link, decisions, challenges, outcome, reflection)")
                tw_import = gr.Button("Import CSV")

                def on_save(member, full, date_, stream, cat, task, contrib, ev, link,
                            decisions, challenges, outcome, reflection):
                    if not member or not task.strip():
                        return "Member and task are required.", _contrib_df()
                    rec = {"member": member, "full_name": full, "date": date_, "workstream": stream,
                           "category": cat, "task": task, "contribution": contrib, "evidence": ev,
                           "evidence_link": link, "decisions": decisions, "challenges": challenges,
                           "outcome": outcome, "reflection": reflection}
                    save_contribution(DB, rec)
                    return f"Saved contribution for {member}: {task}", _contrib_df()

                tw_save.click(on_save, inputs=[tw_member, tw_full, tw_date, tw_stream, tw_cat, tw_task,
                                               tw_contrib, tw_evidence, tw_link, tw_decisions,
                                               tw_challenges, tw_outcome, tw_reflection],
                              outputs=[tw_status, tw_df])
                tw_refresh.click(lambda: _contrib_df(), inputs=[], outputs=[tw_df])

                def on_import(file):
                    if file is None:
                        return "Choose a CSV file first.", _contrib_df()
                    path = file.name if hasattr(file, "name") else str(file)
                    n = 0
                    with open(path, newline="", encoding="utf-8") as fh:
                        for row in csv.DictReader(fh):
                            rec = {k: (row.get(k, "") or "") for k in CONTRIBUTION_FIELDS}
                            if rec.get("member") and rec.get("task"):
                                save_contribution(DB, rec)
                                n += 1
                    return f"Imported {n} contribution record(s).", _contrib_df()

                tw_import.click(on_import, inputs=[tw_file], outputs=[tw_status, tw_df])

            # ---- F. Export Center ---------------------------------------------------
            with gr.Tab("F. Export Center"):
                gr.Markdown("Exports are generated from the actual records in shape.db. "
                            "Report drafts flag missing evidence and reflections — they never invent work.")
                with gr.Row():
                    exp_contrib = gr.Button("Export contributions CSV")
                    exp_eval = gr.Button("Export evaluation CSV")
                    exp_prog = gr.Button("Export progress CSV")
                exp_file = gr.File(label="Download the exported file")
                gen_btn = gr.Button("Generate 4 individual report drafts + team summary", variant="primary")
                gen_out = gr.Textbox(label="Generated drafts (also written as .md files)", lines=20)
                hw_msg = gr.Textbox(label="Hardware mock — test message (JSON)",
                                    value='{"device_id": "pi-zero-01", "event": "lesson_start", "lesson_id": "L1", "action": "next", "timestamp": "2026-11-06T09:00:00"}',
                                    lines=3)
                hw_btn = gr.Button("Test hardware message")
                hw_out = gr.Textbox(label="Hardware mock result", lines=3)

                def export_csv(kind):
                    import json as _json
                    if kind == "contributions":
                        rows = get_contributions(DB)
                        path = "SHAPE_contributions.csv"
                        fields = CONTRIBUTION_FIELDS
                    elif kind == "evaluation":
                        rows = get_attempts(DB)
                        path = "SHAPE_evaluation.csv"
                        fields = ["learner_id", "lesson_id", "passed", "hints_used", "reward", "strategy"]
                    else:
                        rows = get_attempts(DB)
                        path = "SHAPE_progress.csv"
                        fields = ["learner_id", "lesson_id", "passed", "hints_used", "reward", "strategy"]
                    with open(path, "w", newline="", encoding="utf-8") as fh:
                        w = csv.DictWriter(fh, fieldnames=fields)
                        w.writeheader()
                        for r in rows:
                            w.writerow({k: r.get(k, "") for k in fields})
                    return path

                exp_contrib.click(lambda: export_csv("contributions"), inputs=[], outputs=[exp_file])
                exp_eval.click(lambda: export_csv("evaluation"), inputs=[], outputs=[exp_file])
                exp_prog.click(lambda: export_csv("progress"), inputs=[], outputs=[exp_file])

                def on_generate():
                    rows = get_contributions(DB)
                    parts = []
                    for member in MEMBER_NAMES:
                        draft = generate_report_draft(member, rows)
                        fname = f"SHAPE_Report_Draft_{member.replace(' ', '_')}.md"
                        with open(fname, "w", encoding="utf-8") as fh:
                            fh.write(draft)
                        parts.append(draft)
                    summary = team_evidence_summary(rows)
                    with open("SHAPE_Team_Evidence_Summary.md", "w", encoding="utf-8") as fh:
                        fh.write(summary)
                    parts.append(summary)
                    return "\n\n---\n\n".join(parts)

                gen_btn.click(on_generate, inputs=[], outputs=[gen_out])

                def on_hw(msg):
                    try:
                        data = json.loads(msg)
                    except Exception as e:
                        return f"Invalid JSON: {e}"
                    return str(handle_device_message(data))

                hw_btn.click(on_hw, inputs=[hw_msg], outputs=[hw_out])

        # Pre-fill the Learning Lab with Lesson 1 on first load.
        demo.load(lambda: (LESSONS[0]["concept"], LESSONS[0]["example"], LESSONS[0]["exercise"],
                           LESSONS[0]["concept_check_q"]),
                  inputs=[], outputs=[concept_box, example_box, exercise_box, check_q])
    return demo


demo = build_demo()
print('SHAPE demo built. Run the last cell to launch it.')


## Evaluation, hardware and the learning pathway
**Evaluation:** pre-test (5 questions) → lessons → post-test (same 5 questions) → delayed retention check. Measured: correctness, completion, hint dependency, attempts per exercise, pre→post improvement. Learner IDs are anonymised. The chart in the Adaptive Learning tab is **synthetic demo data, clearly labelled SYNTHETIC DEMO ONLY** — never present it as real student results. Real results come only from actual records in `shape.db`.
**Learning pathway:** Python fundamentals (the 6 lessons here) → deep learning foundations → neural networks/CNNs → sequence/NLP → transformers/attention → autoencoders/generative → diffusion → reinforcement learning → advanced agents, planning and orchestration (the 12 HCC resources, in that order).
**Hardware (mock mode, no hardware required):** the central LLM and orchestration run on Colab/a computer. Pi Zero 2 W / Pico W / CYD handhelds only send small events (`device_id`, `event`, `lesson_id`, `action`, `timestamp`) and receive short prompts, progress and feedback — they never run the LLM. Test it in the Export Center tab with the "Test hardware message" button. See the Hardware Integration Plan (separate document) for the full device plan and checklist.


In [ ]:
# Last cell — Launch the SHAPE Agent 16 app.
# In Colab this opens the interface below. Set SHAPE_LAUNCH=0 to skip launching
# (used when testing the notebook automatically).
import os
if os.environ.get("SHAPE_LAUNCH", "1") == "1":
    demo.launch(share=False, theme=gr.themes.Soft())
else:
    print("Launch skipped (SHAPE_LAUNCH=0). demo is built and ready.")
